# To Search Elasticsearch

This script is designed to be a template for elasticsearch instances

In [ ]:
import sys
sys.path.append('..')
from credentials import *

from elasticsearch_utils import *

import pandas as pd
import numpy as np
import re

import json

import random

from medcat.cat import CAT

data_path = "../data/"
raw_data_path = data_path+'raw_data/'

## Elastic Search

### Login and Initialise

In [ ]:
# --- Connect to Elasticsearch ---
es = connect_elasticsearch(hosts=hosts, username=username, password=password, api_key=True)

### Check the list of Indices and Columns

In [ ]:
# --- Explore Indices ---
print("🔍 Available Indices:")
for i in list(es.indices.get_alias().keys()):
    print("-", i)

In [ ]:
# --- Explore Fields in a Given Index ---
index = ['cancer_docs', 'observations', 'medical_history', 'notes', 'noting', 'letters', 'lab_results', 'carenotes']  # 🔧 Set your index name here

if isinstance(index, list):
    print(f"\n📑 Unique Fields for the selected Indexes:")
    fields = []
    for idx in index:
        mapping = get_field_mapping(es, idx)
        for field in mapping[idx]["mappings"]["properties"]:
            fields.append(field)
    fields = list(set(fields))
    fields.sort()
    for field in fields:
        print("-", field)
elif '*' in index:
    pass
else:
    print(f"\n📑 Fields in index: {index}")
    mapping = get_field_mapping(es, index)
    for field in mapping[index]["mappings"]["properties"]:
        print("-", field)

### Create Inclusion Patient Lists

In [ ]:
inclusion_patients_df = pd.read_csv(os.path.join(data_path, "chronic_kidney_disease_refined_inclusion_patients.csv"))[['master_person_id', 'patient_identifier3', 'patient_identifier4', 'patient_identifier2', 'postcode']]

In [ ]:
id_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier4']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            id_number.append(i)

In [ ]:
for idx, row in enumerate(inclusion_patients_df['patient_identifier3']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            try:
                id_number.append(int(i))
            except:
                pass

In [ ]:
for idx, row in enumerate(inclusion_patients_df['patient_identifier2']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            id_number.append(int(i))

### Set parameters and define columns of interest

Select your fields and list in order of output columns

In [ ]:
# --- Define Query Parameters ---
columns = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3',
           'document_Name', 'document_CreatedWhen', 'document_Content']  # 🔧 List of fields you'd like to return (or leave empty to return all)

### Build query

For further information on [how to build a query can be found here](https://www.elastic.co/guide/en/elasticsearch/reference/current/query-dsl.html)

Further information on [free text string queries can be found here](https://www.elastic.co/guide/en/elasticsearch/reference/current/query-dsl-simple-query-string-query.html)


### Search and Save Results

In [ ]:
n = 65536

In [ ]:
df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(id_number, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "must": [
                    {
                        "query_string": {
                            "query": "document_Content : smok* OR document_Content : alcoh* OR document_Content : occupation OR document_Content : lives*",
                            "analyze_wildcard": True,
                            "time_zone": "Europe/London"
                            }
                        }
                    ],
                "filter": [
                    {
                        "bool": {
                            "should": [
                                {"terms": {"patient_identifier1": sub_list}},
                                {"terms": {"patient_identifier3": sub_list}},
                                {"terms": {"patient_identifier2": sub_list}}
                            ],
                            "minimum_should_match": 1
                        }
                    }
                ]
            }
        }
    }

    temp_df = es_docs_to_df(es, index=index, query=query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        df = pd.concat([df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {df.shape[0]:,}\n")

    i+=1

In [ ]:
df.to_csv(raw_data_path+'elasticsearch_search_hits/initial_behaviour_extract.csv', index=False)

### Targetted Occupation Search

In [ ]:
index = "emergency"

columns = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3', 'document_CreatedWhen', 'patient_Occupation']  # 🔧 List of fields you'd like to return (or leave empty to return all)

In [ ]:
n = 65536

In [ ]:
occupation_df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(id_number, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    occupation_query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "filter": [
                    {
                        "bool": {
                            "should": [
                                {"terms": {"patient_identifier1": sub_list}},
                                {"terms": {"patient_identifier3": sub_list}},
                                {"terms": {"patient_identifier2": sub_list}}
                            ],
                            "minimum_should_match": 1
                        }
                    },
                    {
                        "exists": {
                            "field": "patient_Occupation"
                        }
                    }
                ]
            }
        }
    }

    temp_df = es_docs_to_df(es, index=index, query=occupation_query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(occupation_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        occupation_df = pd.concat([occupation_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {occupation_df.shape[0]:,}\n")

    i+=1

In [ ]:
patient_identifer3_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[1].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            patient_identifer3_dict[nhs_num]=row[0]

occupation_df['masterPersonId3'] = occupation_df['patient_identifier3'].map(patient_identifer3_dict)

In [ ]:
patient_identifer4_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[2].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            patient_identifer4_dict[nhs_num]=row[0]

occupation_df['masterPersonId4'] = occupation_df['patient_identifier4'].map(patient_identifer4_dict)

In [ ]:
patient_identifer2_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[3].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            patient_identifer2_dict[nhs_num]=row[0]

occupation_df['masterPersonId2'] = occupation_df['patient_identifier2'].map(patient_identifer2_dict)

In [ ]:
occupation_df.insert(0, 'master_person_id', occupation_df['masterPersonId3'].combine_first(occupation_df['masterPersonId4']).combine_first(occupation_df['masterPersonId2']))

occupation_df = occupation_df[occupation_df['document_CreatedWhen']<'2026-01-01']

occupation_df['document_CreatedDate'] = pd.to_datetime(occupation_df['document_CreatedWhen'], format='mixed').dt.date

occupation_df = occupation_df[['master_person_id', 'document_CreatedDate', 'patient_Occupation']].sort_values(by=['master_person_id', 'document_CreatedDate'])
occupation_df = occupation_df.groupby(['master_person_id', 'document_CreatedDate']).first().reset_index()

occupation_df = occupation_df.sort_values(by=['master_person_id', 'patient_Occupation', 'document_CreatedDate'])
occupation_df = occupation_df.groupby(['master_person_id', 'patient_Occupation']).first().reset_index()

occupation_df.head()

In [ ]:
occupation_df.to_csv(raw_data_path+'elasticsearch_search_hits/occupation_extract.csv', index=False)

### Postcode Search

In [ ]:
id_number = []
for idx, row in enumerate(inclusion_patients_df[inclusion_patients_df['postcode'].isna()]['patient_identifier4']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            id_number.append(i)

In [ ]:
for idx, row in enumerate(inclusion_patients_df[inclusion_patients_df['postcode'].isna()]['patient_identifier3']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            try:
                id_number.append(int(i))
            except:
                pass

In [ ]:
for idx, row in enumerate(inclusion_patients_df[inclusion_patients_df['postcode'].isna()]['patient_identifier2']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            id_number.append(int(i))

In [ ]:
index = "all"

columns = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3', 'patient_AddressPostalCode']  # 🔧 List of fields you'd like to return (or leave empty to return all)

In [ ]:
n = 65536

In [ ]:
postcode_df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(id_number, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    postcode_query = {
        "size": 0,
        "query": {
            "bool": {
                "filter": [
                    {
                        "bool": {
                            "should": [
                                {"terms": {"patient_identifier1": sub_list}},
                                {"terms": {"patient_identifier3": sub_list}},
                                {"terms": {"patient_identifier2": sub_list}}
                            ],
                            "minimum_should_match": 1
                        }
                    },
                    {
                        "exists": {
                            "field": "patient_AddressPostalCode"
                        }
                    }
                ]
            }
        },
        "aggs": {
            "distinct_postcodes": {
                "terms": {
                    "field": "patient_AddressPostalCode.keyword",
                    "size": 10000
                    }
                }
            }
        }

    temp_df = es_docs_to_df(es, index=index, query=postcode_query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(postcode_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        postcode_df = pd.concat([postcode_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {postcode_df.shape[0]:,}\n")

    i+=1

In [ ]:
patient_identifer3_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[1].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            patient_identifer3_dict[nhs_num]=row[0]

postcode_df['masterPersonId3'] = postcode_df['patient_identifier3'].map(patient_identifer3_dict)

In [ ]:
patient_identifer4_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[2].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            patient_identifer4_dict[nhs_num]=row[0]

postcode_df['masterPersonId4'] = postcode_df['patient_identifier4'].map(patient_identifer4_dict)

In [ ]:
patient_identifer2_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[3].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            patient_identifer2_dict[nhs_num]=row[0]

postcode_df['masterPersonId2'] = postcode_df['patient_identifier2'].map(patient_identifer2_dict)

In [ ]:
cols = ['master_person_id', 'patient_AddressPostalCode']

postcode_df.insert(0, 'master_person_id', postcode_df['masterPersonId3'].combine_first(postcode_df['masterPersonId4']).combine_first(postcode_df['masterPersonId2']))

postcode_df = postcode_df[cols].drop_duplicates().reset_index(drop=True)

postcode_df.head()

In [ ]:
postcode_df.to_csv(raw_data_path+'elasticsearch_search_hits/postcode_extract.csv', index=False)

# Process Search Results

In [ ]:
inclusion_patients_df = pd.read_csv(os.path.join(data_path, "chronic_kidney_disease_refined_inclusion_patients.csv"))

## Process

### Load Search Results

In [ ]:
df = pd.read_csv(raw_data_path+'elasticsearch_search_hits/initial_behaviour_extract.csv')

df = df[df['document_Content'].notna()]
df['document_Content'] = df['document_Content'].astype(str)
df['document_Content'] = df['document_Content'].str.replace('\r', '', regex=False)

df['patient_identifier2'] = df['patient_identifier2'].apply(lambda x: x if pd.isna(x) else str(x)).apply(lambda x: x if pd.isna(x) else x[:-2])
df['patient_identifier3'] = df['patient_identifier3'].apply(lambda x: x if pd.isna(x) else str(x)).apply(lambda x: x if pd.isna(x) else x[:-2])

### Connect to Unique Patient Idetifier

In [ ]:
patient_identifer3_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[1].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            patient_identifer3_dict[nhs_num]=row[0]

df['masterPersonId3'] = df['patient_identifier3'].map(patient_identifer3_dict)

In [ ]:
patient_identifer4_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[2].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            patient_identifer4_dict[nhs_num]=row[0]

df['masterPersonId4'] = df['patient_identifier4'].map(patient_identifer4_dict)

In [ ]:
patient_identifer2_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[3].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            patient_identifer2_dict[nhs_num]=row[0]

df['masterPersonId2'] = df['patient_identifier2'].map(patient_identifer2_dict)

In [ ]:
df.insert(0, 'master_person_id', df['masterPersonId3'].combine_first(df['masterPersonId4']).combine_first(df['masterPersonId2']))

df = df[['master_person_id', 'patient_identifier4', 'patient_identifier2', 'patient_identifier3', 'document_Name', 'document_CreatedWhen', 'document_Content']]

del patient_identifer3_dict, patient_identifer4_dict, patient_identifer2_dict

df.head()

### Extract Relevant Social & Behavioural Data

#### Smoking Status

In [ ]:
def smoking_string_regex(text):
    if pd.isna(text) or text is None:
        return np.NaN

    text = str(text)

    pattern1 = r'''((?:current|non|ex|denies|deny|does|has|never)[\s-].{0,12}(?:smok|vape|cigarette)\w{0,3}\??)'''

    pattern2 = r'''((?:smok|vape|cigarette)\w{0,3}[\n\t]{1,}[tfcn]\w{1,7}\b)'''

    pattern3 = r'''((?:current|currently|previously).{1,5}(?:smok|vape|cigarette)\w{0,3}.{1,5}[yn]\w{0,2})'''

    pattern4 = r'''(\([-+]\).{1,3}smok\w{0,3})'''

    pattern5 = r'''(do you smok\w{0,3}.[\n\t]*?\w{1,3}\b)'''

    pattern6 = r'''(other details[\n\t]*?smok\w{0,3}\b)'''

    pattern7 = r'''(smoking history:.[a-z/]{1,})'''

    pattern8 = r'''(\w*\ssmoking history(?!:))'''

    pattern9 = r'''(smok\w{1,3}:.*?\w+\b)'''

    all_matches = []

    for pattern in [pattern1, pattern2, pattern3, pattern4, pattern5, pattern6, pattern7, pattern8, pattern9]:
        regex = re.compile(pattern, re.IGNORECASE)
        matches = regex.findall(text)

        all_matches.extend(matches)
    
    cleaned_matches = []
    seen = set()
    
    for match in all_matches:
        clean_match = re.sub(r'\s+', ' ', match)
        clean_match = clean_match.title()
        if clean_match and clean_match not in seen:
            cleaned_matches.append(clean_match)
            seen.add(clean_match)

    if len(cleaned_matches) == 1:
        return cleaned_matches[0]
    elif len(cleaned_matches) == 0:
        return np.nan
    else:
        return ', '.join(cleaned_matches)

In [ ]:
df['smoking_extract'] = df['document_Content'].apply(lambda x: smoking_string_regex(x))

#### Alcohol Consumption

In [ ]:
def alcohol_string_regex(text):
    if pd.isna(text) or text is None:
        return np.NaN

    text = str(text)

    pattern1 = r'''((?:does|rare|no|doesn't|deny|denies|nil|never|moderate)\s(?:\b\w*?\b ){0,5}alcohol(?: intake)?)'''

    pattern2 = r'''(alcohol(?: use| drink\w{0,3})?\s?(?::|-)\s*?(?:\b\w*\b(?:\s|/)?){1,})'''

    pattern3 = r'''(do you drink alcohol.[\n\t]*?\w{1,3}\b)'''

    pattern4 = r'''(alcohol consumption[\n\d\w/]{1,15})'''

    pattern5 = r'''(u(?:nits)?(?: of)?(?: consumed| alcohol)?(?: per)?(?: |/)(?:week|day)[\s\d\w/-]{1,15})'''

    pattern6 = r'''(\([-+]\).{1,3}alcohol(?:.*?\([\w\s]*?\))?)'''

    pattern7 = r'''([\s\d\w/-]{1,15} u(?:nits)?(?: of)?(?: consumed| alcohol)?(?: per)?(?: |/)(?:week|day))'''

    all_matches = []

    for pattern in [pattern1, pattern2, pattern3, pattern4, pattern5, pattern6, pattern7]:
        regex = re.compile(pattern, re.IGNORECASE)
        matches = regex.findall(text)

        all_matches.extend(matches)
    
    cleaned_matches = []
    seen = set()
    
    for match in all_matches:
        clean_match = re.sub(r'\s+', ' ', match)
        clean_match = clean_match.title()
        if clean_match and clean_match not in seen:
            cleaned_matches.append(clean_match)
            seen.add(clean_match)

    if len(cleaned_matches) == 1:
        return cleaned_matches[0]
    elif len(cleaned_matches) == 0:
        return np.nan
    else:
        return ', '.join(cleaned_matches)

In [ ]:
df['alcohol_extract'] = df['document_Content'].apply(lambda x: alcohol_string_regex(x))

#### Occupation

In [ ]:
def occupation_string_regex(text):
    if pd.isna(text) or text is None:
        return np.NaN

    text = str(text)
    
    pattern1 = r'''((?:former )?occupation:?(?: )?(?:\b\w*\b ?)*)'''

    pattern2 = r'''(occupation\n{1,2}(?:\b\w*\b ?)*)'''

    all_matches = []

    for pattern in [pattern1, pattern2]:
        regex = re.compile(pattern, re.IGNORECASE)
        matches = regex.findall(text)

        all_matches.extend(matches)
    
    cleaned_matches = []
    seen = set()
    
    for match in all_matches:
        clean_match = re.sub(r'\s+', ' ', match)
        clean_match = clean_match.title()
        if clean_match and clean_match not in seen:
            cleaned_matches.append(clean_match)
            seen.add(clean_match)

    if len(cleaned_matches) == 1:
        return cleaned_matches[0]
    elif len(cleaned_matches) == 0:
        return np.nan
    else:
        return ', '.join(cleaned_matches)

In [ ]:
df['occupation_extract'] = df['document_Content'].apply(lambda x: occupation_string_regex(x))

#### Living Status

In [ ]:
def living_situation_string_regex(text):
    if pd.isna(text) or text is None:
        return np.NaN

    text = str(text)

    pattern1 = r'''(lives.*?(?:\.|\,|\n))'''

    all_matches = []

    for pattern in [pattern1]:
        regex = re.compile(pattern, re.IGNORECASE)
        matches = regex.findall(text)

        all_matches.extend(matches)
    
    cleaned_matches = []
    seen = set()
    
    for match in all_matches:
        clean_match = re.sub(r'\s+', ' ', match)
        clean_match = clean_match.title()
        if clean_match and clean_match not in seen:
            cleaned_matches.append(clean_match)
            seen.add(clean_match)

    if len(cleaned_matches) == 1:
        return cleaned_matches[0]
    elif len(cleaned_matches) == 0:
        return np.nan
    else:
        return ', '.join(cleaned_matches)

In [ ]:
df['living_situation_extract'] = df['document_Content'].apply(lambda x: living_situation_string_regex(x))

### Filter for Presence of above Extracts

In [ ]:
social_behavioural_df = df[(df['smoking_extract'].notna())|(df['alcohol_extract'].notna())|(df['occupation_extract'].notna())|(df['living_situation_extract'].notna())]

del df, inclusion_patients_df

social_behavioural_df = social_behavioural_df.drop(columns=['document_Content']).sort_values(by=['master_person_id', 'document_CreatedWhen']).reset_index(drop=True)

social_behavioural_df.to_csv(raw_data_path+'elasticsearch_search_hits/social_behavioural_data.csv', index=False)

# Clean and Structure Final Results

In [ ]:
social_behavioural_df = pd.read_csv(raw_data_path+'elasticsearch_search_hits/social_behavioural_data.csv')

social_behavioural_df['patient_identifier2'] = social_behavioural_df['patient_identifier2'].apply(lambda x: x if pd.isna(x) else str(x)).apply(lambda x: x if pd.isna(x) else x[:-2])
social_behavioural_df['patient_identifier3'] = social_behavioural_df['patient_identifier3'].apply(lambda x: x if pd.isna(x) else str(x)).apply(lambda x: x if pd.isna(x) else x[:-2])

social_behavioural_df['document_CreatedWhen'] = pd.to_datetime(social_behavioural_df['document_CreatedWhen'], format='mixed')
social_behavioural_df.insert(5, 'document_CreatedWhenDate', social_behavioural_df['document_CreatedWhen'].dt.date)

social_behavioural_df = social_behavioural_df[social_behavioural_df['master_person_id'].notna()]
social_behavioural_df = social_behavioural_df.drop(columns=['document_CreatedWhen'])
social_behavioural_df = social_behavioural_df.drop_duplicates().reset_index(drop=True)

social_behavioural_df.head()

In [ ]:
social_behavioural_df.shape

## Smoking Status

In [ ]:
def smoking_status(text):
    text = str(text)
    text = text.lower()

    if text == 'nan':
        return np.NaN
    elif 'unknown' in text:
        return np.NaN
    elif 'ex' in text or 'previous' in text\
                      or 'stopped' in text\
                      or 'has given up' in text\
                      or 'has smoked' in text\
                      or 'quit' in text\
                      or 'former' in text\
                      or 'has been' in text:
        return 'Ex-Smoker'
    elif 'never' in text or 'false' in text\
                         or 'denies' in text\
                         or 'denied' in text\
                         or '? no' in text\
                         or 'non' in text\
                         or 'nil' in text\
                         or 'n/a' in text\
                         or 'decline' in text\
                         or ('smok' in text and 'no' in text)\
                         or 'does no' in text\
                         or '(-)' in text:
        return 'Non-Smoker'
    elif 'occasional' in text:
        return 'Occasional Smoker'
    elif 'current smoker' in text or 'true' in text\
                                  or 'yes' in text\
                                  or 'does continue' in text\
                                  or 'does smok' in text\
                                  or '(+)' in text\
                                  or 'social' in text\
                                  or 'still' in text\
                                  or 'has cut down' in text\
                                  or 'has reduce' in text\
                                  or 'has start' in text\
                                  or 'smokes: cigars' in text:
        return 'Current Smoker'
    else:
        return np.NaN

In [ ]:
social_behavioural_df['smoking_status'] = social_behavioural_df['smoking_extract'].apply(lambda x: smoking_status(x))

In [ ]:
social_behavioural_df['smoking_status'].value_counts(dropna=False)

## Alcohol Use

In [ ]:
def alcohol_use(text):
    text = str(text)
    text = text.lower()

    if 'nan' in text:
        return np.NaN
    elif 'alcoh' not in text:
        return np.NaN
    elif 'occasional' in text:
        return 'Occasional'
    elif 'no more than' in text or 'no record of initial alcohol' in text:
        return np.NaN
    elif '? no' in text or 'nil' in text\
                        or 'does not' in text\
                        or ': no' in text\
                        or 'never' in text\
                        or "doesn't" in text\
                        or 'denies' in text\
                        or 'no alcohol' in text\
                        or 'n/a' in text:
        return 'Does not Drink Alcohol'
    elif '? yes' in text or 'alcohol consumption yes' in text\
                         or ': yes' in text:
        return 'Drinks Alcohol'
    elif len(re.findall(r'\d{2}-\w{2,3}-\d{2,4}', text))>=1:
        return np.NaN
    elif len(re.findall(r'\d{1,2}(?: ?- ?\d{1,2})?\s?u(?:nits)?(?: consumed| of alcohol)? ?per ?week', text, flags=re.IGNORECASE))>=1:
        return ', '.join(list(set(re.findall(r'(\d{1,2}(?: ?- ?\d{1,2})?\s?u(?:nits)?(?: consumed| of alcohol)? ?per ?week)', text, flags=re.IGNORECASE))))
    elif len(re.findall(r'\d{1,2}(?: ?- ?\d{1,2})?\s?u(?:nits)?[/ ]??week', text, flags=re.IGNORECASE))>=1:
        return ', '.join(list(set(re.findall(r'(\d{1,2}(?: ?- ?\d{1,2})?\s?u(?:nits)?[/ ]??week)', text, flags=re.IGNORECASE))))
    elif len(re.findall(r'u(?:nits)?[/ ]?week\s?\d{1,2}(?: ?- ?\d{1,2})?', text, flags=re.IGNORECASE))>=1:
        return ', '.join(list(set(re.findall(r'(u(?:nits)?[/ ]?week\s?\d{1,2}(?: ?- ?\d{1,2})?)', text, flags=re.IGNORECASE))))
    elif len(re.findall(r'u(?:nits)?(?: consumed)? ? per ?week\s?\d{1,2}(?: ?- ?\d{1,2})?', text, flags=re.IGNORECASE))>=1:
        return ', '.join(list(set(re.findall(r'(u(?:nits)?(?: consumed| of alcohol)? ?per ?week\s?\d{1,2}(?: ?- ?\d{1,2})?)', text, flags=re.IGNORECASE))))
    elif len(re.findall(r'\d{1,2}(?: ?- ?\d{1,2})?\s?u(?:nits)?(?: consumed| of alcohol)? ?per ?day', text, flags=re.IGNORECASE))>=1:
        return ', '.join(list(set(re.findall(r'(\d{1,2}(?: ?- ?\d{1,2})?\s?u(?:nits)?(?: consumed)? ?per ?day)', text, flags=re.IGNORECASE))))
    elif len(re.findall(r'u(?:nits)?(?: consumed| of alcohol)? ? per ?day\s?\d{1,2}(?: ?- ?\d{1,2})?', text, flags=re.IGNORECASE))>=1:
        return ', '.join(list(set(re.findall(r'(u(?:nits)?(?: consumed| of alcohol)? ?per ?day\s?\d{1,2}(?: ?- ?\d{1,2})?)', text, flags=re.IGNORECASE))))
    elif 'not on file' in text or text == 'alcohol consumption '\
                               or text == 'units consumed per week if alcohol '\
                               or text == 'alcohol consumption date descripti'\
                               or text == 'units consumed per week '\
                               or ': smoking' in text\
                               or ': action required' in text\
                               or text == 'ally / drinks _ units of alcohol/week'\
                               or ': advice given':
        return np.NaN
    else:
        return np.NaN

In [ ]:
social_behavioural_df['alcohol_use'] = social_behavioural_df['alcohol_extract'].apply(lambda x: alcohol_use(x))

In [ ]:
def alcohol_classification(text):
    text = str(text)
    text = text.lower()

    if text == 'nan':
        return np.NaN  
    elif 'week' in text:
        result = list(set([int(x) if x != '' else np.NaN for x in re.findall(r'\d*', text)]))
        minimum = 1e6
        for i in result:
            if i == 'nan':
                pass
            if i < minimum:
                minimum = i
        return minimum
    elif 'day' in text:
        result = list(set([int(x)*7 if x != '' else np.NaN for x in re.findall(r'\d*', text)]))
        minimum = 1e6
        for i in result:
            if i == 'nan':
                pass
            if i < minimum:
                minimum = i
        return minimum
    else:
        return text.title()

In [ ]:
social_behavioural_df['alcohol_classification'] = social_behavioural_df['alcohol_use'].apply(lambda x: alcohol_classification(x))

In [ ]:
social_behavioural_df['alcohol_classification'].value_counts(dropna=False)

## Occupation

#### Populate with Strucutred Elastic Data

In [ ]:
occupation_df = pd.read_csv(raw_data_path+'elasticsearch_search_hits/occupation_extract.csv')

occupation_df.columns = ['master_person_id', 'occupation_extract', 'document_CreatedWhenDate']

In [ ]:
social_behavioural_df = pd.concat([social_behavioural_df, occupation_df])

In [ ]:
def occupation(text):
    text = str(text)
    text = text.lower()

    if text == 'occupation' or text == 'occupation:'\
                            or text == 'occupation '\
                            or text == 'occupation code'\
                            or 'occupation, occupation' in text\
                            or 'occupation , occupation' in text\
                            or 'occupation:, occupation' in text\
                            or 'occupation: , occupation' in text\
                            or text == 'occupation: '\
                            or text == 'nan'\
                            or 'occupation therapy' in text\
                            or text == 'nil'\
                            or text == 'none'\
                            or text == 'n/a':
        result = np.NaN
    elif 'retired' in text or text == 'rtd'\
                           or text == 'rt'\
                           or text == 'r-t'\
                           or text =='ret'\
                           or text == 'rt.'\
                           or text == 'r t'\
                           or 'elderly' in text or\
                           'ellderly' in text or\
                           text == 'r/t' or \
                           'ret/' in text:
        result = 'Retired'
    elif 'disabled' in text:
        result = 'Disabled'
    elif 'house wife' in text or text == 'hw'\
                              or 'housewife' in text\
                              or text == 'h/w'\
                              or text == 'h / w'\
                              or 'mother' in text\
                              or 'h/wife' in text\
                              or 'mum' in text:
        result = 'Housewife'
    elif 'details with patient' in text or 'not known' in text\
                                        or text == 'nk'\
                                        or text == 'unknown'\
                                        or text == 'u/k'\
                                        or text == 'does not want to say'\
                                        or text == 'do not want to say':
        result = np.NaN
    elif 'unemployed' in text or 'nil at present' in text\
                              or 'made redundant' in text\
                              or 'not work' in text\
                              or text == 'u/e'\
                              or 'u/e' in text\
                              or text == 'u./e'\
                              or text == 'ue'\
                              or text == 'n.e.'\
                              or text == 'u.e'\
                              or text == 'u-e'\
                              or text == 'u / e'\
                              or text == 'un-employed'\
                              or 'u/e ' in text:
        result = 'Unemployed'
    elif 'off sick' in text:
        result = 'Off Sick'
    elif 'student' in text:
        result = 'Student'
    elif 'self' in text or text == 's/e':
        result = 'Self Employed'
    elif len(re.findall(r'occupation:\s?\w*', text, flags=re.IGNORECASE))>=1:
        result = ', '.join([x.title() for x in re.findall(r'occupation:\s?(\w*(?:\s?\w*)*)', text, flags=re.IGNORECASE)])
        if result:
            result = 'Employed'
    elif len(re.findall(r'^[a-z]*$', text, flags=re.IGNORECASE))>=1:
        result = 'Employed'
    elif len(re.findall(r'^occupation.*', text, flags=re.IGNORECASE|re.DOTALL))>=1:
        result = np.NaN
    elif len(re.findall(r'\d+', text, flags=re.IGNORECASE))>=1:
        result = np.NaN
    elif len(re.findall(r'^[`,.-]+$', text, flags=re.IGNORECASE))>=1:
        result = np.NaN
    elif len(re.findall(r'^[a-z]* [a-z]*$|^[a-z]* [a-z]* [a-z]*$', text, flags=re.IGNORECASE))>=1:
        result = 'Employed'
    elif 'employed' in text:
        result = 'Employed'
    else:
        result = np.NaN

    if 'part time' in str(result).lower() or 'part time' in text\
                                          or 'p/t' in text\
                                          or 'p/time' in text\
                                          or 'part-time' in text:
        result = 'Part Time'
    if result == '':
        result = np.NaN
    
    return result

In [ ]:
social_behavioural_df['occupation'] = social_behavioural_df['occupation_extract'].apply(lambda x: occupation(x))

In [ ]:
social_behavioural_df['occupation'].value_counts(dropna=False)

## Living Status

In [ ]:
def living_status(text):
    text = str(text)
    text = text.lower()

    if text == 'nan' or text == 'lives.'\
                     or text == 'lives '\
                     or text == 'lives,':
        return np.NaN
    elif 'alon' in text or 'independ' in text\
                        or 'on own' in text\
                        or 'on her own' in text\
                        or 'on his own' in text\
                        or 'by herself' in text\
                        or 'by himself' in text\
                        or 'aloe' in text\
                        or 'alione' in text\
                        or 'by self' in text\
                        or 'a lone' in text:
        return 'Lives Alone'
    elif len(re.findall(r'lives\s?(?:.*?[wirth\+]{1,4}\s?(?:his\s|her\s)?)?.*?(?:w.fe|famil|husb...|daug..er|son|child|partner|kid|sister|mother|father|brother|gran|relative|friend|girlfriend|boyfriend|m.m|dad|parent|ne.hew|niece|cousin|spouse)', text, flags=re.IGNORECASE)) >=1 or 'flatmate' in text:
        return 'Lives Accompanied'
    elif 'sheltered' in text or 'sheltred' in text:
        return 'Lives in Sheltered Accommodation'
    elif 'care home' in text or 'nursing home' in text\
                             or 'care facility' in text:
        return 'Lives in Care Home'
    elif 'supported' in text:
        return 'Lives in Supported Living'
    elif text == 'lives at home,' or text == 'lives at home '\
                                  or text == 'lives in hostel '\
                                  or text == 'lives in a flat '\
                                  or text == 'lives in flat '\
                                  or text == 'lives in flat'\
                                  or text == 'lives in flat.'\
                                  or text == 'lives in 1 bed flat '\
                                  or text == 'lives in 1 bed flat.'\
                                  or text == 'lives at home.'\
                                  or text == 'lives in a residential home':
        return np.NaN
    else:
        return np.NaN

In [ ]:
social_behavioural_df['living_status'] = social_behavioural_df['living_situation_extract'].apply(lambda x: living_status(x))

In [ ]:
social_behavioural_df['living_status'].value_counts(dropna=False)

### Refine Social and Behaviourl Data

In [ ]:
social_behavioural_refined_df = social_behavioural_df[['master_person_id', 'document_CreatedWhenDate', 'smoking_status', 'alcohol_classification', 'occupation', 'living_status']]

In [ ]:
smoking_status_filter = (social_behavioural_refined_df['smoking_status'].isna())
alcohol_classification_filter = (social_behavioural_refined_df['alcohol_classification'].isna())
occupation_filter = (social_behavioural_refined_df['occupation'].isna())
living_status_filter = (social_behavioural_refined_df['living_status'].isna())

full_filter = (smoking_status_filter&alcohol_classification_filter&occupation_filter&living_status_filter)

In [ ]:
social_behavioural_refined_df = social_behavioural_refined_df[~full_filter]

social_behavioural_refined_df['document_CreatedWhenDate'] = pd.to_datetime(social_behavioural_refined_df['document_CreatedWhenDate']).dt.date

social_behavioural_refined_df = social_behavioural_refined_df.reset_index(drop=True).sort_values(by=['master_person_id', 'document_CreatedWhenDate']).drop_duplicates()

In [ ]:
agg = {}

for col in social_behavioural_refined_df.columns[2:]:
    agg[col] = 'first'

social_behavioural_refined_df = social_behavioural_refined_df.groupby(['master_person_id', 'document_CreatedWhenDate']).agg(agg).reset_index()

social_behavioural_refined_df.head()

## Save

In [ ]:
# --- Save Results ---
path_to_results = raw_data_path
file_name = "20251205_social_behavioural_search_results.csv"

social_behavioural_refined_df.to_csv(f"{path_to_results}/{file_name}", index=False)
print("✅ Results saved.")

# Sandbox